In [ ]:
"""
Ticket Classification — Two-Stage Domain-Adaptive RoBERTa  
====================================================================
Stage 1 : Domain-adaptive MLM pretraining on unlabeled IT tickets
Stage 2 : Supervised classifier fine-tuning on labeled tickets
"""

import os, json, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch._dynamo
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report,
)
from transformers import (
    RobertaTokenizerFast,
    RobertaForMaskedLM,
    RobertaForSequenceClassification,
    DataCollatorForLanguageModeling,
    get_cosine_schedule_with_warmup,
)

# ─────────────────────────────────────────────────────────────────────────────
# Configuration Loader
# ─────────────────────────────────────────────────────────────────────────────
import yaml

DEFAULT_CONFIG = {
    "stage1": {
        "base_model": "roberta-base",
        "text_col": "ticket_description",
        "max_length": 128,
        "batch_size": 16,
        "epochs": 3,
        "lr": 5e-5,
        "weight_decay": 0.01,
        "mlm_probability": 0.15,
        "val_split": 0.05,
    },
    "stage2": {
        "text_col": "ticket_description",
        "label_col": "ticket_category",
        "max_length": 128,
        "batch_size": 16,
        "epochs": 10,
        "lr": 2e-5,
        "weight_decay": 0.01,
        "label_smoothing": 0.10,
        "patience": 3,
    },
}

def load_config(config_path: str = "config.yaml") -> dict:
    """
    Load configuration from YAML file.
    If file doesn't exist, return default configuration.
    
    Args:
        config_path (str): Path to the YAML configuration file.
        
    Returns:
        dict: Configuration dictionary with stage1 and stage2 settings.
    """
    if not os.path.exists(config_path):
        return DEFAULT_CONFIG.copy()
    
    with open(config_path, "r", encoding="utf-8") as f:
        user_config = yaml.safe_load(f) or {}
    
    # Merge user config with defaults (user values override defaults)
    config = DEFAULT_CONFIG.copy()
    for stage in ["stage1", "stage2"]:
        if stage in user_config:
            config[stage].update(user_config[stage])
    
    return config


def save_default_config(config_path: str = "config.yaml") -> str:
    """
    Save the default configuration to a YAML file for reference.
    
    Args:
        config_path (str): Path to save the configuration file.
        
    Returns:
        str: The path where the config was saved.
    """
    with open(config_path, "w", encoding="utf-8") as f:
        yaml.dump(DEFAULT_CONFIG, f, default_flow_style=False, sort_keys=False)
    return config_path

OUTPUT_DIR = "roberta_2stage_output"
PLOTS_DIR  = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# Encoding-resilient CSV reader
# ─────────────────────────────────────────────────────────────────────────────
_ENCODINGS_TO_TRY = ["utf-8-sig", "utf-8", "latin-1", "cp1252", "iso-8859-1"]


def read_csv_safe(path_or_buffer, **kwargs):
    """
    Wrapper around pd.read_csv that automatically detects the file encoding.

    Tries a sequence of common encodings (utf-8-sig, utf-8, latin-1, cp1252,
    iso-8859-1) and falls back to utf-8 with errors='replace' so that *any*
    file can be read without crashing.

    For in-memory file-like objects (e.g. Streamlit UploadedFile) the buffer
    is rewound (``seek(0)``) between attempts.

    Args:
        path_or_buffer: File path (str / pathlib.Path) or file-like object.
        **kwargs: Extra keyword arguments forwarded to ``pd.read_csv``.

    Returns:
        pd.DataFrame: The parsed CSV data.
    """
    is_file_like = hasattr(path_or_buffer, "read")

    for enc in _ENCODINGS_TO_TRY:
        try:
            if is_file_like:
                path_or_buffer.seek(0)
            return pd.read_csv(path_or_buffer, encoding=enc, **kwargs)
        except (UnicodeDecodeError, UnicodeError):
            continue
        except Exception:
            # Non-encoding errors (EmptyDataError, ParserError, etc.) should
            # propagate immediately — they won't be fixed by changing encoding.
            raise

    # Last resort: read with replacement characters so we never crash.
    if is_file_like:
        path_or_buffer.seek(0)
    return pd.read_csv(path_or_buffer, encoding="utf-8", errors="replace", **kwargs)


# ─────────────────────────────────────────────────────────────────────────────
# Seed & device
# ─────────────────────────────────────────────────────────────────────────────
def set_seed(seed: int = 42):
    """Set random seed for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)


# ─────────────────────────────────────────────────────────────────────────────
# Step 0: Extract and deduplicate descriptions for Stage 1
# ─────────────────────────────────────────────────────────────────────────────
def prepare_unlabeled_csv(file_names: list, output_path: str = "only_descriptions.csv"):
    """
    Reads ticket_description from all source CSVs, deduplicates on
    stripped text, and saves.
    
    Args:
        file_names (list): List of file paths to process.
        output_path (str): Path to save the deduplicated CSV.
        
    Returns:
        str: The output_path where the CSV was saved.
    """
    frames = []
    for f in file_names:
        # Explicit check for non-CSV file format
        if not str(f).lower().endswith('.csv'):
            raise ValueError(f"Invalid file format: '{f}'. Only CSV format is supported.")
        try:
            frames.append(read_csv_safe(f, usecols=["ticket_description"]))
        except Exception as e:
            print(f"Warning: could not load {f}: {e}")

    if not frames:
        raise ValueError("No objects to concatenate: None of the provided CSV files contained valid data.")
    combined     = pd.concat(frames, ignore_index=True)
    col          = combined["ticket_description"].dropna().astype(str).str.strip()
    before_count = len(col)

    col = col[~col.duplicated(keep="first")]
    col = col[col != ""]
    after_count = len(col)
    col.to_csv(output_path, index=False)

    print(f"Unlabeled CSV created: {output_path}")
    print(f"  Records before dedup: {before_count}")
    print(f"  Duplicates removed  : {before_count - after_count}")
    print(f"  Records saved       : {after_count}")
    return output_path


# ─────────────────────────────────────────────────────────────────────────────
# Datasets
# ─────────────────────────────────────────────────────────────────────────────
class UnlabeledTicketDataset(Dataset):
    def __init__(self, texts: list, tokenizer, max_length: int = 128):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].size(0)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }


class LabeledTicketDataset(Dataset):
    def __init__(self, texts: list, labels: list, tokenizer, max_length: int = 128):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def save_json(obj: dict, path: str):
    """
    Save a dictionary object to a JSON file with UTF-8 encoding.
    
    Args:
        obj (dict): The dictionary to save.
        path (str): The destination file path.
        
    Returns:
        None
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_unlabeled_texts(csv_path: str, text_col: str = "ticket_description") -> list:
    """
    Load and clean unlabeled text data from a CSV file.
    
    Args:
        csv_path (str): Path to the CSV file.
        text_col (str): Name of the column containing the text.
        
    Returns:
        list: A list of cleaned string texts.
    """
    df = read_csv_safe(csv_path)
    df = df.dropna(subset=[text_col]).copy()
    df[text_col] = df[text_col].astype(str).str.strip()
    return df.loc[df[text_col] != "", text_col].tolist()


def load_labeled_data(
    csv_path: str,
    text_col: str  = "ticket_description",
    label_col: str = "ticket_category",
) -> pd.DataFrame:
    """
    Load labeled data from CSV, validate columns, clean text/labels, and remove duplicates.
    
    Args:
        csv_path (str): Path to the labeled CSV file.
        text_col (str): Name of the text description column.
        label_col (str): Name of the target category column.
        
    Returns:
        pd.DataFrame: Cleaned DataFrame with no NaNs, standardized casing, and no duplicates.
    """
    # TC111: Explicit check for non-CSV file format
    if not str(csv_path).lower().endswith('.csv'):
        raise ValueError(f"Invalid file format: '{csv_path}'. Only CSV format is supported.")
        
    try:
        df = read_csv_safe(csv_path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=[text_col, label_col])
    
    # TC004 & TC005: Explicit check for required columns before any processing
    missing_cols = []
    if text_col not in df.columns:
        missing_cols.append(text_col)
    if label_col not in df.columns:
        missing_cols.append(label_col)
    if missing_cols:
        raise KeyError(f"Missing required column(s): {', '.join(missing_cols)}")
    
    before = len(df)
    df = df.dropna(subset=[text_col, label_col]).copy()
    df[text_col]  = df[text_col].astype(str).str.strip().str.lower()
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[(df[text_col] != "") & (df[label_col] != "")]
    df = df.drop_duplicates(subset=[text_col, label_col])
    after = len(df)
    print(f"Labeled data — loaded: {before}, after cleaning: {after}, removed: {before-after}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Stage 1: Domain-adaptive MLM pretraining
# ─────────────────────────────────────────────────────────────────────────────
def train_stage1_mlm(
    unlabeled_csv: str,
    output_dir: str   = "./roberta_it_stage1",
    config_path: str  = "config.yaml",
    # The following can override config values if provided explicitly
    base_model: str   = None,
    text_col: str     = None,
    max_length: int   = None,
    batch_size: int   = None,
    epochs: int       = None,
    lr: float         = None,
    weight_decay: float = None,
    mlm_probability: float = None,
    val_split: float  = None,
) -> str:
    """
    Stage 1: Domain-adaptive MLM pretraining on unlabeled IT tickets.
    
    Loads configuration from config.yaml if present, otherwise uses defaults.
    Explicit function arguments override config values when provided.
    
    Args:
        unlabeled_csv (str): Path to unlabeled CSV file.
        output_dir (str): Directory to save the trained model.
        config_path (str): Path to YAML config file (default: "config.yaml").
        base_model (str): Pre-trained model name (overrides config).
        text_col (str): Text column name (overrides config).
        max_length (int): Maximum sequence length (overrides config).
        batch_size (int): Training batch size (overrides config).
        epochs (int): Number of training epochs (overrides config).
        lr (float): Learning rate (overrides config).
        weight_decay (float): Weight decay (overrides config).
        mlm_probability (float): MLM masking probability (overrides config).
        val_split (float): Validation split ratio (overrides config).
        
    Returns:
        str: Path to the output directory.
    """
    # Load config and apply overrides
    cfg = load_config(config_path)["stage1"]
    
    base_model = base_model or cfg["base_model"]
    text_col = text_col or cfg["text_col"]
    max_length = int(max_length or cfg["max_length"])
    batch_size = int(batch_size or cfg["batch_size"])
    epochs = int(epochs if epochs is not None else cfg["epochs"])
    lr = float(lr if lr is not None else cfg["lr"])
    weight_decay = float(weight_decay if weight_decay is not None else cfg["weight_decay"])
    mlm_probability = float(mlm_probability if mlm_probability is not None else cfg["mlm_probability"])
    val_split = float(val_split if val_split is not None else cfg["val_split"])
    
    os.makedirs(output_dir, exist_ok=True)

    tokenizer = RobertaTokenizerFast.from_pretrained(base_model)
    model     = RobertaForMaskedLM.from_pretrained(base_model).to(device)
    collator  = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=True, mlm_probability=mlm_probability
    )

    texts = load_unlabeled_texts(unlabeled_csv, text_col=text_col)

    n_val   = max(1, int(len(texts) * val_split))
    if len(texts) <= 1: 
        n_val = 0
    random.shuffle(texts)
    val_texts   = texts[:n_val]
    train_texts = texts[n_val:]
    print(f"\nStage 1 MLM — train: {len(train_texts):,}  val: {len(val_texts):,}")

    train_loader = DataLoader(
        UnlabeledTicketDataset(train_texts, tokenizer, max_length),
        batch_size=batch_size, shuffle=True, collate_fn=collator,
    )
    val_loader = DataLoader(
        UnlabeledTicketDataset(val_texts, tokenizer, max_length),
        batch_size=batch_size, shuffle=False, collate_fn=collator,
    )

    total_steps = len(train_loader) * epochs
    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.10 * total_steps),
        num_training_steps=total_steps,
    )

    prev_train_loss = None
    for epoch in range(epochs):
        # ── Train ──
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Stage1 Epoch {epoch+1}/{epochs}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        avg_train = epoch_loss / len(train_loader)
        train_ppl = math.exp(avg_train)

        # ── Val perplexity ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch     = {k: v.to(device) for k, v in batch.items()}
                val_loss += model(**batch).loss.item()
        avg_val = val_loss / len(val_loader)
        val_ppl = math.exp(avg_val)

        trend = ""
        if prev_train_loss is not None:
            drop = prev_train_loss - avg_train
            trend = (f"  ↓ {drop:.4f}" if drop > 0.005 else
                     f"  ✗ stalled ({drop:.4f})")
        print(f"Stage1 Epoch {epoch+1} | "
              f"Train Loss: {avg_train:.4f} (PPL {train_ppl:.2f}) | "
              f"Val Loss: {avg_val:.4f} (PPL {val_ppl:.2f}){trend}")
        prev_train_loss = avg_train

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\nStage 1 model saved → {output_dir}")
    return output_dir


# ─────────────────────────────────────────────────────────────────────────────
# Evaluation helper
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_classifier(
    model: RobertaForSequenceClassification,
    loader: DataLoader,
    label_smoothing: float = 0.1,
) -> dict:
    """
    Returns loss, accuracy, weighted_f1, macro_f1, preds, labels.
    """
    model.eval()
    loss_fn    = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            batch        = {k: v.to(device) for k, v in batch.items()}
            labels_batch = batch.pop("labels")
            outputs      = model(**batch)
            total_loss  += loss_fn(outputs.logits, labels_batch).item()
            preds        = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    return {
        "loss":        total_loss / len(loader),
        "accuracy":    accuracy_score(all_labels, all_preds),
        "f1":          f1_score(all_labels, all_preds, average="micro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
        "macro_f1":    f1_score(all_labels, all_preds, average="macro"),
        "preds":       all_preds,
        "labels":      all_labels,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Stage 2: Classifier fine-tuning
# ─────────────────────────────────────────────────────────────────────────────
def train_stage2_classifier(
    labeled_csv: str,
    stage1_model_dir: str  = "./roberta_it_stage1",
    output_dir: str        = "./roberta_it_ticket_classifier",
    text_col: str          = "ticket_description",
    label_col: str         = "ticket_category",
    max_length: int        = 128,
    batch_size: int        = 16,
    epochs: int            = 10,
    lr: float              = 2e-5,
    weight_decay: float    = 0.01,
    label_smoothing: float = 0.10,
    patience: int          = 3,
) -> tuple:
    """
    Returns (output_dir, label2id, id2label, train_history).
    """
    os.makedirs(output_dir, exist_ok=True)

    # TC094: Handle corrupt CSV input with informative error
    try:
        df = load_labeled_data(labeled_csv, text_col=text_col, label_col=label_col)
    except pd.errors.ParserError as e:
        raise ValueError(f"Failed to parse CSV file '{labeled_csv}': {e}")
    except Exception as e:
        if "ParserError" in str(type(e)) or "parsing" in str(e).lower():
            raise ValueError(f"Failed to parse CSV file '{labeled_csv}': {e}")
        raise

    labels   = sorted(df[label_col].unique().tolist())
    
    # TC036: Explicit check for insufficient class diversity
    if len(labels) < 2:
        raise ValueError(
            f"Insufficient class diversity: found {len(labels)} unique class(es) '{labels}'. "
            f"Stratified split requires at least 2 classes."
        )
    
    label2id = {lbl: i for i, lbl in enumerate(labels)}
    id2label = {i: lbl for lbl, i in label2id.items()}
    num_cls  = len(label2id)
    df["label_id"] = df[label_col].map(label2id)

    set_seed(42)

    # 80/10/10 split
    try:
        df_tmp, test_df = train_test_split(
            df, test_size=0.10, random_state=42, stratify=df["label_id"]
        )
        train_df, val_df = train_test_split(
            df_tmp, test_size=0.1111, random_state=42, stratify=df_tmp["label_id"]
        )
    except ValueError as e:
        if "test_size" in str(e) and "number of classes" in str(e):
            raise ValueError("Insufficient class diversity for stratified split")
        raise

    total_samples = len(train_df) + len(val_df) + len(test_df)
    print(f"\n{'='*60}")
    print(f"📊 DATA SPLIT SUMMARY")
    print(f"{'='*60}")
    print(f"  Total cleaned samples : {total_samples:,}")
    print(f"  Train                 : {len(train_df):,} ({100*len(train_df)/total_samples:.1f}%)")
    print(f"  Validation            : {len(val_df):,} ({100*len(val_df)/total_samples:.1f}%)")
    print(f"  Test                  : {len(test_df):,} ({100*len(test_df)/total_samples:.1f}%)")
    print(f"  Number of classes     : {num_cls}")
    print(f"  Classes               : {labels}")
    print(f"{'='*60}")

    # Per-class distribution in each split
    print(f"\n  {'Category':<15} {'Train':>8} {'Val':>8} {'Test':>8}")
    print(f"  {'-'*15} {'-'*8} {'-'*8} {'-'*8}")
    for lbl in labels:
        lid = label2id[lbl]
        tr_c = len(train_df[train_df["label_id"] == lid])
        va_c = len(val_df[val_df["label_id"] == lid])
        te_c = len(test_df[test_df["label_id"] == lid])
        print(f"  {lbl:<15} {tr_c:>8,} {va_c:>8,} {te_c:>8,}")
    print()

    tokenizer = RobertaTokenizerFast.from_pretrained(stage1_model_dir)
    model     = RobertaForSequenceClassification.from_pretrained(
        stage1_model_dir,
        num_labels=num_cls,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    # Class-weighted loss
    train_labels_np = train_df["label_id"].values
    class_weights = compute_class_weight(
        "balanced", classes=np.unique(train_labels_np), y=train_labels_np
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print(f"Class weights: {dict(zip([id2label[i] for i in range(num_cls)], class_weights.round(4)))}")

    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights_tensor, label_smoothing=label_smoothing
    )

    def make_loader(frame, shuffle):
        return DataLoader(
            LabeledTicketDataset(
                frame[text_col].tolist(), frame["label_id"].tolist(),
                tokenizer, max_length,
            ),
            batch_size=batch_size, shuffle=shuffle,
        )

    train_loader = make_loader(train_df, shuffle=True)
    val_loader   = make_loader(val_df,   shuffle=False)
    test_loader  = make_loader(test_df,  shuffle=False)

    # Scheduler
    expected_epochs = min(epochs, patience + 1)
    total_steps     = len(train_loader) * expected_epochs
    optimizer       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler       = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.10 * total_steps),
        num_training_steps=total_steps,
    )

    best_val_loss     = float("inf")
    epochs_no_improve = 0
    train_history     = []

    print(f"\n{'='*60}")
    print(f"🚀 Stage 2: Fine-tuning on {len(train_df):,} labeled tickets")
    print(f"{'='*60}\n")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        train_preds, train_labels = [], []

        for batch in tqdm(train_loader, desc=f"Stage2 Epoch {epoch+1}/{epochs}"):
            batch        = {k: v.to(device) for k, v in batch.items()}
            labels_batch = batch.pop("labels")
            optimizer.zero_grad()
            outputs      = model(**batch)
            
            # Explicit shape assertions to catch malformed inputs early
            assert outputs.logits.shape[0] == labels_batch.shape[0], \
                f"Batch size mismatch: logits {outputs.logits.shape[0]} vs labels {labels_batch.shape[0]}"
            assert outputs.logits.shape[1] == num_cls, \
                f"Logits dimension mismatch: {outputs.logits.shape[1]} != num_classes {num_cls}"
            
            loss         = loss_fn(outputs.logits, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels_batch.cpu().numpy())

        train_loss = epoch_loss / len(train_loader)
        train_acc  = accuracy_score(train_labels, train_preds)
        train_wf1  = f1_score(train_labels, train_preds, average="weighted")
        train_mf1  = f1_score(train_labels, train_preds, average="macro")

        val_metrics = evaluate_classifier(model, val_loader, label_smoothing)

        gap = val_metrics["loss"] - train_loss
        fit_status = (
            f"⚠ overfit gap={gap:.4f}" if gap > 0.15 else
            f"mild overfit gap={gap:.4f}" if gap > 0.05 else
            f"good fit gap={gap:.4f}"
        )

        print(
            f"\nEpoch {epoch+1:2d}/{epochs}\n"
            f"  Train → Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  "
            f"W-F1: {train_wf1:.4f}  M-F1: {train_mf1:.4f}\n"
            f"  Val   → Loss: {val_metrics['loss']:.4f}  Acc: {val_metrics['accuracy']:.4f}  "
            f"W-F1: {val_metrics['weighted_f1']:.4f}  M-F1: {val_metrics['macro_f1']:.4f}\n"
            f"  [{fit_status}]"
        )

        train_history.append({
            "epoch":        epoch + 1,
            "train_loss":   round(train_loss,                  6),
            "train_acc":    round(train_acc,                   6),
            "train_wf1":    round(train_wf1,                   6),
            "train_mf1":    round(train_mf1,                   6),
            "val_loss":     round(val_metrics["loss"],         6),
            "val_acc":      round(val_metrics["accuracy"],     6),
            "val_wf1":      round(val_metrics["weighted_f1"],  6),
            "val_mf1":      round(val_metrics["macro_f1"],     6),
        })

        if val_metrics["loss"] < best_val_loss:
            best_val_loss     = val_metrics["loss"]
            epochs_no_improve = 0
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            save_json(label2id, os.path.join(output_dir, "label2id.json"))
            save_json(id2label, os.path.join(output_dir, "id2label.json"))
            print(f"  → ✅ Saved best model (val loss {best_val_loss:.4f})")
        else:
            epochs_no_improve += 1
            print(f"  → No improvement ({epochs_no_improve}/{patience})")
            if epochs_no_improve >= patience:
                print("  → 🛑 Early stopping triggered.")
                break

    # Guard — save model if nothing was ever saved
    if not os.path.exists(os.path.join(output_dir, "config.json")):
        print("  ⚠ No checkpoint saved. Saving current model as fallback.")
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        save_json(label2id, os.path.join(output_dir, "label2id.json"))
        save_json(id2label, os.path.join(output_dir, "id2label.json"))

    # ══════════════════════════════════════════════════════════════
    # FINAL EVALUATION — Best Model Only
    # ══════════════════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print(f"📊 FINAL EVALUATION (Best Model)")
    print(f"{'='*60}")

    best_model = RobertaForSequenceClassification.from_pretrained(output_dir).to(device)
    ordered_names = [id2label[i] for i in sorted(id2label.keys())]

    train_final = evaluate_classifier(best_model, train_loader, label_smoothing)
    val_final   = evaluate_classifier(best_model, val_loader,   label_smoothing)
    test_final  = evaluate_classifier(best_model, test_loader,  label_smoothing)

    # Print summary table
    print(f"\n  {'Split':<12} {'Samples':>8} {'Loss':>8} {'Acc':>8} {'W-F1':>8} {'M-F1':>8}")
    print(f"  {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
    for name, metrics, n in [
        ("Train", train_final, len(train_df)),
        ("Validation", val_final, len(val_df)),
        ("Test", test_final, len(test_df)),
    ]:
        print(
            f"  {name:<12} {n:>8,} {metrics['loss']:>8.4f} "
            f"{metrics['accuracy']:>8.4f} {metrics['weighted_f1']:>8.4f} "
            f"{metrics['macro_f1']:>8.4f}"
        )

    # ── Process splits: Build JSONs, then generate PNGs from those JSONs ─────
    splits_data = [
        ("Training",   train_final, len(train_df)),
        ("Validation", val_final,   len(val_df)),
        ("Testing",    test_final,  len(test_df)),
    ]

    per_class_all = {} 

    for split_name, split_metrics, n_samples in splits_data:
        s_preds  = split_metrics["preds"]
        s_labels = split_metrics["labels"]

        # Console report
        print(f"\n{'─'*60}")
        print(f"📋 {split_name.upper()} Classification Report ({n_samples:,} samples)")
        print(f"{'─'*60}")
        report_text = classification_report(
            s_labels, s_preds, target_names=ordered_names,
        )
        print(report_text)

        # JSON classification report (Saved to master eval_results.json ONLY)
        report_dict = classification_report(
            s_labels, s_preds, target_names=ordered_names, output_dict=True,
        )
        
        # Accumulate for eval_results.json
        per_class_all[f"per_class_{split_name.lower()}_metrics"] = {
            cat: {
                "precision": round(report_dict[cat]["precision"], 4),
                "recall":    round(report_dict[cat]["recall"],    4),
                "f1":        round(report_dict[cat]["f1-score"],  4),
                "support":   int(report_dict[cat]["support"]),
            }
            for cat in ordered_names if cat in report_dict
        }

        # Save Confusion Matrix JSON first (Source of Truth)
        cm = confusion_matrix(s_labels, s_preds)
        cm_json_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{split_name.lower()}.json")
        save_json({
            "_split": split_name,
            "labels": ordered_names,
            "matrix": cm.tolist(),
        }, cm_json_path)
        
        # Generate PNG strictly from the JSON we just saved
        _plot_confusion(cm_json_path)

    # ── Save Master eval_results.json ──
    eval_results = {
        "model":          "RoBERTa 2-Stage (domain-adaptive)",
        "num_classes":    num_cls,
        "class_names":    ordered_names,
        "data_split": {
            "total_samples":  total_samples,
            "train_samples":  len(train_df),
            "val_samples":    len(val_df),
            "test_samples":   len(test_df),
            "train_pct":      round(100 * len(train_df) / total_samples, 1),
            "val_pct":        round(100 * len(val_df) / total_samples, 1),
            "test_pct":       round(100 * len(test_df) / total_samples, 1),
        },
        "best_val_loss":  round(best_val_loss, 6),
        "train_metrics": {
            "accuracy":    round(train_final["accuracy"],    6),
            "f1":          round(train_final["f1"],          6),
            "weighted_f1": round(train_final["weighted_f1"], 6),
            "macro_f1":    round(train_final["macro_f1"],    6),
            "loss":        round(train_final["loss"],        6),
        },
        "val_metrics": {
            "accuracy":    round(val_final["accuracy"],    6),
            "f1":          round(val_final["f1"],          6),
            "weighted_f1": round(val_final["weighted_f1"], 6),
            "macro_f1":    round(val_final["macro_f1"],    6),
            "loss":        round(val_final["loss"],        6),
        },
        "test_metrics": {
            "accuracy":    round(test_final["accuracy"],    6),
            "f1":          round(test_final["f1"],          6),
            "weighted_f1": round(test_final["weighted_f1"], 6),
            "macro_f1":    round(test_final["macro_f1"],    6),
            "loss":        round(test_final["loss"],        6),
        },
        **per_class_all,
        "train_history":  train_history,
        "hyperparameters": {
            "max_length":       max_length,
            "learning_rate":    lr,
            "batch_size":       batch_size,
            "epochs":           epochs,
            "weight_decay":     weight_decay,
            "label_smoothing":  label_smoothing,
            "patience":         patience,
            "warmup_ratio":     0.10,
        },
    }
    save_json(eval_results, os.path.join(OUTPUT_DIR, "eval_results.json"))
    
    # Generate Learning Curves PNG strictly from the JSON we just saved
    _plot_learning_curves(
        os.path.join(OUTPUT_DIR, "eval_results.json"), 
        os.path.join(PLOTS_DIR, "stage2_learning_curves.png")
    )
    
    print(f"\n✅ Eval results saved → {OUTPUT_DIR}/eval_results.json")

    return output_dir, label2id, id2label, train_history

# ─────────────────────────────────────────────────────────────────────────────
# Plot helpers (STRICTLY JSON-FIRST)
# ─────────────────────────────────────────────────────────────────────────────
def _plot_learning_curves(json_path: str, save_path: str):
    """Generates learning curves PNG strictly by reading the eval_results.json."""
    if not os.path.exists(json_path):
        return
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    history = data.get("train_history", [])
    if not history:
        return
        
    epochs   = [r["epoch"]      for r in history]
    tl       = [r["train_loss"] for r in history]
    vl       = [r["val_loss"]   for r in history]
    ta       = [r["train_acc"]  for r in history]
    va       = [r["val_acc"]    for r in history]
    tmf      = [r["train_mf1"]  for r in history]
    vmf      = [r["val_mf1"]    for r in history]
    twf      = [r["train_wf1"]  for r in history]
    vwf      = [r["val_wf1"]    for r in history]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("RoBERTa 2-Stage — Stage 2 Training History",
                 fontsize=14, fontweight="bold", y=1.01)

    axes[0, 0].plot(epochs, tl, "o-", color="#2563eb", lw=1.8, label="Train loss")
    axes[0, 0].plot(epochs, vl, "s--", color="#dc2626", lw=1.8, label="Val loss")
    axes[0, 0].set_title("Loss", fontsize=12, fontweight="bold")
    axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend(fontsize=9); axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(epochs, [v*100 for v in ta], "o-", color="#2563eb", lw=1.8, label="Train acc")
    axes[0, 1].plot(epochs, [v*100 for v in va], "s--", color="#dc2626", lw=1.8, label="Val acc")
    axes[0, 1].set_title("Accuracy", fontsize=12, fontweight="bold")
    axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Accuracy (%)")
    axes[0, 1].set_ylim(0, 100)
    axes[0, 1].legend(fontsize=9); axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(epochs, [v*100 for v in tmf], "o-", color="#2563eb", lw=1.8, label="Train M-F1")
    axes[1, 0].plot(epochs, [v*100 for v in vmf], "s--", color="#dc2626", lw=1.8, label="Val M-F1")
    axes[1, 0].set_title("Macro F1", fontsize=12, fontweight="bold")
    axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("F1 (%)")
    axes[1, 0].set_ylim(0, 100)
    axes[1, 0].legend(fontsize=9); axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(epochs, [v*100 for v in twf], "o-", color="#2563eb", lw=1.8, label="Train W-F1")
    axes[1, 1].plot(epochs, [v*100 for v in vwf], "s--", color="#dc2626", lw=1.8, label="Val W-F1")
    axes[1, 1].set_title("Weighted F1", fontsize=12, fontweight="bold")
    axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("F1 (%)")
    axes[1, 1].set_ylim(0, 100)
    axes[1, 1].legend(fontsize=9); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"📈 Learning curves saved → {save_path}")


def _plot_confusion(json_path: str):
    """Generates Confusion Matrix PNG strictly by reading the saved JSON."""
    if not os.path.exists(json_path):
        return
        
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    labels = data["labels"]
    cm = np.array(data["matrix"])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    split_name = data.get("_split", "Test")
    save_path = os.path.join(PLOTS_DIR, f"confusion_matrix_{split_name.lower()}.png")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Confusion Matrix — RoBERTa 2-Stage ({split_name})",
                 fontsize=13, fontweight="bold", y=1.01)
    for ax, matrix, title, fmt in [
        (axes[0], cm,      "Raw counts",               "d"),
        (axes[1], cm_norm, "Row-normalised (recall %)", ".2f"),
    ]:
        sns.heatmap(matrix, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=labels, yticklabels=labels,
                    linewidths=0.4, linecolor="white", ax=ax,
                    cbar_kws={"shrink": 0.8})
        ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=30); ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"📊 Confusion matrix saved → {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Load predictor
# ─────────────────────────────────────────────────────────────────────────────
def load_predictor(model_dir: str = "./roberta_it_ticket_classifier"):
    """Load model, tokenizer, and id2label once at startup."""
    tokenizer = RobertaTokenizerFast.from_pretrained(model_dir)
    model     = RobertaForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    # JSON-FIRST: Load label2id and reverse it instead of keeping a duplicate id2label file
    label2id_path = os.path.join(model_dir, "label2id.json")
    with open(label2id_path, encoding="utf-8") as f:
        label2id = json.load(f)
    id2label = {v: k for k, v in label2id.items()}

    return tokenizer, model, id2label


# ─────────────────────────────────────────────────────────────────────────────
# Predict
# ─────────────────────────────────────────────────────────────────────────────
def predict_ticket(
    text: str,
    tokenizer,
    model: RobertaForSequenceClassification,
    id2label: dict,
    max_length: int = 128,
) -> dict:
    """
    Simple prediction — always returns the top predicted category.
    No uncertainty gating. Confidence, entropy, and margin are returned
    as informational metrics only. Top 3 predictions are also returned.
    """
    model.eval()
    enc = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=max_length, return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        
        # Explicit shape assertion for inference
        assert enc["input_ids"].shape[1] == max_length, \
            f"Input sequence length {enc['input_ids'].shape[1]} != max_length {max_length}"
        assert outputs.logits.shape[1] == len(id2label), \
            f"Logits shape {outputs.logits.shape[1]} != num_classes {len(id2label)}"
            
        probs   = torch.softmax(outputs.logits, dim=1)
        top_num = min(3, probs.size(1))
        top_k   = torch.topk(probs, top_num)
        
        conf    = top_k.values[0][0].item()
        pred_id = top_k.indices[0][0].item()
        margin  = (conf - top_k.values[0][1].item()) if probs.size(1) > 1 else 1.0
        entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=1).item()
        
        top_3_preds = []
        for i in range(top_num):
            cat = id2label[top_k.indices[0][i].item()]
            c_score = top_k.values[0][i].item()
            top_3_preds.append({"category": cat, "confidence": round(c_score, 4)})

    return {
        "text":               text,
        "predicted_category": id2label[pred_id],
        "confidence_score":   round(conf,    4),
        "entropy":            round(entropy, 4),
        "margin":             round(margin,  4),
        "top_3_predictions":  top_3_preds,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    SOURCE_FILES = [
        "/all_tickets_processed_improved_v3.csv",
        "/cleaned_service_desk_balanced_final.csv",
        "/service_desk_strong_classification_data.csv",
        "/generateditsm/generateditsm20k_data.csv",
    ]
    LABELED_CSV   = "/cleaned_service_desk_balanced_final.csv"
    UNLABELED_CSV = "/only_descriptions.csv"

    # ── Stage 0: build unlabeled CSV ──────────────────────────────────────────
    prepare_unlabeled_csv(SOURCE_FILES, output_path=UNLABELED_CSV)

    # ── Stage 1: domain-adaptive MLM pretraining ──────────────────────────────
    stage1_dir = train_stage1_mlm(
        unlabeled_csv=UNLABELED_CSV,
        output_dir="./roberta_it_stage1",
        base_model="roberta-base",
        text_col="ticket_description",
        max_length=128,
        batch_size=16,
        epochs=3,
        lr=5e-5,
        val_split=0.05,
    )

    # ── Stage 2: classifier fine-tuning ──────────────────────────────────────
    stage2_dir, label2id, id2label, train_history = train_stage2_classifier(
        labeled_csv=LABELED_CSV,
        stage1_model_dir=stage1_dir,
        output_dir="./roberta_it_ticket_classifier",
        text_col="ticket_description",
        label_col="ticket_category",
        max_length=128,
        batch_size=16,
        epochs=10,
        lr=2e-5,
        label_smoothing=0.10,
        patience=3,
    )

    # ── Load for inference ────────────────────────────────────────────────────
    tokenizer, model, id2label = load_predictor(stage2_dir)

    # ── Sample predictions ────────────────────────────────────────────────────
    sample_texts = [
        "VPN login keeps failing with authentication error after password reset.",
        "Outlook calendar events are not syncing with mobile devices.",
        "The air conditioning in the server room is leaking.",
        "asdfghjkl qwerty uiop zxcvbnm.",
        "New employee onboarding — need accounts and equipment set up by Monday.",
        "Shared drive showing only 2 GB free — team unable to upload files.",
    ]

    print(f"\n{'='*60}")
    print(f"🔮 Sample Predictions")
    print(f"{'='*60}")
    for text in sample_texts:
        r = predict_ticket(text, tokenizer, model, id2label)
        print(f"\n  Text      : {text}")
        print(f"  Predicted : {r['predicted_category']}")
        print(f"  Conf      : {r['confidence_score']:.3f}  "
              f"Margin: {r['margin']:.3f}  Entropy: {r['entropy']:.3f}")

    print(f"\n✅ All done. Artefacts in: {OUTPUT_DIR}")

2026-03-31 10:31:10.213994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774953070.413062      33 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774953070.468948      33 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Unlabeled CSV created: /kaggle/working/only_descriptions.csv
  Records before dedup: 149515
  Duplicates removed  : 69526
  Records saved       : 79989


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


Epoch  4/10
  Train → Loss: 0.7694  Acc: 0.9501  W-F1: 0.9501  M-F1: 0.9535
  Val   → Loss: 0.7785  Acc: 0.9006  W-F1: 0.9006  M-F1: 0.9029
  [good fit gap=0.0091]
  → No improvement (1/3)


Stage2 Epoch 5/10: 100%|██████████| 2550/2550 [15:42<00:00,  2.71it/s]



Epoch  5/10
  Train → Loss: 0.7614  Acc: 0.9541  W-F1: 0.9541  M-F1: 0.9573
  Val   → Loss: 0.7928  Acc: 0.8992  W-F1: 0.8991  M-F1: 0.9023
  [good fit gap=0.0314]
  → No improvement (2/3)


Stage2 Epoch 6/10: 100%|██████████| 2550/2550 [15:41<00:00,  2.71it/s]



Epoch  6/10
  Train → Loss: 0.7756  Acc: 0.9486  W-F1: 0.9486  M-F1: 0.9523
  Val   → Loss: 0.8213  Acc: 0.8871  W-F1: 0.8872  M-F1: 0.8929
  [good fit gap=0.0457]
  → No improvement (3/3)
  → 🛑 Early stopping triggered.

📊 FINAL EVALUATION (Best Model)

  Split         Samples     Loss      Acc     W-F1     M-F1
  ------------ -------- -------- -------- -------- --------
  Train          40,797   0.6339   0.9537   0.9537   0.9576
  Validation      5,100   0.7730   0.8967   0.8967   0.8998
  Test            5,100   0.7595   0.9065   0.9064   0.9127

────────────────────────────────────────────────────────────
📋 TRAINING Classification Report (40,797 samples)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

      access       0.95      0.94      0.94      7212
       email       0.92      0.97      0.95      2820
    hardware       0.96      0.95      0.95      6794
          hr       0.97      0.98      0.97      6074


In [2]:

TEST_CASES = [
    # ── ACCESS (29) ──────────────────────────────────────────────────────────
    ("User cannot log in to the HR portal after password reset.", "access"),
    ("Need to grant read access to the shared finance drive for new employee.", "access"),
    ("Account locked after multiple failed login attempts.", "access"),
    ("Request to revoke access for terminated employee John Smith.", "access"),
    ("Unable to access SharePoint document library — permission denied error.", "access"),
    ("New contractor requires temporary access badge and system credentials.", "access"),
    ("Admin rights needed on local machine for software installation.", "access"),
    ("Two-factor authentication not working for VPN login.", "access"),
    ("User forgot their Active Directory password and cannot reset it.", "access"),
    ("Cannot access the payroll system — account shows as inactive.", "access"),
    ("Request elevated privileges for database administrator role.", "access"),
    ("Login session keeps timing out after 5 minutes on the web portal.", "access"),
    ("Single sign-on not redirecting correctly to the CRM dashboard.", "access"),
    ("Service account credentials expired, causing batch jobs to fail.", "access"),
    ("Need to add user to security group for project file access.", "access"),
    ("Guest Wi-Fi access required for visiting vendor in conference room B.", "access"),
    ("Account disabled without notice — user cannot log into any systems.", "access"),
    ("Password policy requires special characters but portal rejects them.", "access"),
    ("User transferred to new department, old access needs updating.", "access"),
    ("Remote desktop connection refused — credentials not accepted.", "access"),
    ("API key for internal tool has expired, request renewal.", "access"),
    ("User needs read-only access to production database for reporting.", "access"),
    ("Cannot unlock workstation — smartcard reader not recognising badge.", "access"),
    ("Role-based access not properly applied after user role change in AD.", "access"),
    ("Shared mailbox access request for the support team distribution list.", "access"),
    ("Biometric scanner at server room door not recognising fingerprint.", "access"),
    ("Temporary password sent via SMS not arriving on user's phone.", "access"),
    ("User unable to authenticate to Citrix receiver — error 1030.", "access"),
    ("Access request for new hire starting Monday — all standard applications.", "access"),

    # ── EMAIL (29) ───────────────────────────────────────────────────────────
    ("Outlook keeps crashing when opening attachments larger than 10 MB.", "email"),
    ("Emails sent to external clients bouncing back with delivery failure.", "email"),
    ("Cannot send emails — SMTP authentication error in Outlook.", "email"),
    ("Inbox not syncing on mobile device after OS upgrade.", "email"),
    ("Spam filter blocking legitimate supplier invoices, need whitelist.", "email"),
    ("Distribution list not delivering to all members.", "email"),
    ("Out-of-office auto-reply not triggering for external senders.", "email"),
    ("Email signature disappeared after recent IT policy update.", "email"),
    ("Shared mailbox missing from Outlook profile after migration.", "email"),
    ("Received phishing email, need IT to investigate and block sender.", "email"),
    ("Calendar invites not syncing between Outlook and Google Calendar.", "email"),
    ("Cannot access archived emails from before January 2023.", "email"),
    ("Outlook search returns no results even for recent emails.", "email"),
    ("Email quota full — old emails cannot be archived automatically.", "email"),
    ("New employee needs corporate email account created.", "email"),
    ("Rules and filters in Outlook stopped working after update.", "email"),
    ("Email sent but not appearing in sent folder on mobile.", "email"),
    ("Large email attachment limit needs to be increased for project team.", "email"),
    ("Contacts list not populating when typing in the To field.", "email"),
    ("Email account compromised — request immediate password reset and audit.", "email"),
    ("Delegate access to manager's calendar is not working.", "email"),
    ("Outlook keeps asking for password repeatedly despite correct entry.", "email"),
    ("Need to set up a shared alias address for the customer service team.", "email"),
    ("Emails arriving with garbled characters in subject line (encoding issue).", "email"),
    ("Email forwarding rule set up but forwarded messages not arriving.", "email"),
    ("Unable to open .msg attachments in the web version of Outlook.", "email"),
    ("Migration to Microsoft 365 caused old PST files to become inaccessible.", "email"),
    ("Group email not receiving replies — reply-to address misconfigured.", "email"),
    ("Junk mail folder filling up despite trained spam filter.", "email"),

    # ── HARDWARE (29) ────────────────────────────────────────────────────────
    ("Laptop screen flickering intermittently, especially on battery power.", "hardware"),
    ("Keyboard keys sticking after liquid spill — needs replacement.", "hardware"),
    ("Printer in room 204 jamming frequently on duplex print jobs.", "hardware"),
    ("Desktop PC not powering on — no lights or fan activity.", "hardware"),
    ("Monitor displaying distorted colours after HDMI cable replacement.", "hardware"),
    ("Docking station not detected when laptop is connected via USB-C.", "hardware"),
    ("Hard drive making clicking noises — suspected imminent failure.", "hardware"),
    ("Wireless mouse losing connection every few minutes.", "hardware"),
    ("Webcam not recognised in any USB port after Windows update.", "hardware"),
    ("Laptop battery draining in under 2 hours despite recent replacement.", "hardware"),
    ("Second monitor not being detected after graphics driver update.", "hardware"),
    ("Barcode scanner in warehouse freezing when scanning high-volume items.", "hardware"),
    ("Headset microphone not working — only one-way audio in Teams calls.", "hardware"),
    ("Desktop RAM needs to be upgraded from 8 GB to 16 GB for design work.", "hardware"),
    ("Printer toner cartridge replaced but still printing faded text.", "hardware"),
    ("Laptop overheating and shutting down during long video calls.", "hardware"),
    ("USB hub on workstation not powering connected devices reliably.", "hardware"),
    ("Trackpad on laptop not responding to multi-finger gestures.", "hardware"),
    ("Network switch in server room showing amber fault light.", "hardware"),
    ("Fingerprint reader on laptop not initialising after Windows Hello setup.", "hardware"),
    ("Scanner flatbed producing vertical lines across all scanned documents.", "hardware"),
    ("Conference room TV remote not pairing with the display unit.", "hardware"),
    ("External SSD not recognised on Mac despite working on Windows.", "hardware"),
    ("Power cable for workstation frayed — fire hazard, needs replacement.", "hardware"),
    ("UPS unit beeping — battery backup may need replacing.", "hardware"),
    ("Laptop hinge cracked — lid wobbles when opening and closing.", "hardware"),
    ("Ethernet port on desktop appears damaged — no physical connection.", "hardware"),
    ("Label printer in dispatch not feeding labels correctly.", "hardware"),
    ("Projector in boardroom not displaying laptop screen via HDMI.", "hardware"),

    # ── HR (28) ──────────────────────────────────────────────────────────────
    ("New employee onboarding — need accounts and equipment set up by Monday.", "hr"),
    ("Offboarding request for employee leaving on Friday — revoke all access.", "hr"),
    ("IT system needed for mandatory annual compliance training module.", "hr"),
    ("Employee self-service portal not showing updated payslip for last month.", "hr"),
    ("Maternity leave substitution — temp staff needs system access for 6 months.", "hr"),
    ("Background check system not accessible for new recruitment drive.", "hr"),
    ("HR information system showing incorrect reporting line for new manager.", "hr"),
    ("Request IT equipment for remote worker joining from overseas office.", "hr"),
    ("Performance review system down during appraisal season.", "hr"),
    ("Contractor badge expiry needs extending — still active on project.", "hr"),
    ("Workforce scheduling tool not reflecting updated shift patterns.", "hr"),
    ("New starter laptop needs to be pre-configured with standard software.", "hr"),
    ("Exit interview data not saving in HR portal — form submission error.", "hr"),
    ("Employee attendance system not syncing with payroll software.", "hr"),
    ("HR analytics dashboard permission needed for department manager.", "hr"),
    ("Training platform login credentials not sent to new joiners.", "hr"),
    ("Payroll integration with HR system failing — no data transfer this month.", "hr"),
    ("Time-off request system sending duplicate approval notifications.", "hr"),
    ("Employee records migration to new HR system — data validation errors.", "hr"),
    ("New hire requires IT induction and equipment checklist completion.", "hr"),
    ("Org chart in internal directory not reflecting recent restructure.", "hr"),
    ("Benefits portal showing wrong enrolment status for employee.", "hr"),
    ("HR system audit trail missing entries from last quarter.", "hr"),
    ("Intern accounts need to be created with restricted access profile.", "hr"),
    ("Learning management system not recording completed course credits.", "hr"),
    ("Employee ID badge printing system out of laminate pouches.", "hr"),
    ("Promotion processed in HR system but access rights not updated.", "hr"),
    ("Return-to-work after long-term absence — reactivate dormant account.", "hr"),

    # ── NETWORK (28) ─────────────────────────────────────────────────────────
    ("Office Wi-Fi extremely slow since this morning — affects all users.", "network"),
    ("Cannot connect to VPN from home — connection times out.", "network"),
    ("Intermittent packet loss on wired connection in building 3.", "network"),
    ("DNS resolution failing for internal intranet addresses.", "network"),
    ("Remote site connectivity down — branch office cannot reach HQ systems.", "network"),
    ("Firewall blocking legitimate traffic to cloud storage provider.", "network"),
    ("IP address conflict causing network drop for multiple workstations.", "network"),
    ("Network printer offline despite being powered on and connected.", "network"),
    ("VPN client showing connected but no traffic passing through tunnel.", "network"),
    ("Load balancer misconfiguration causing intermittent 502 errors.", "network"),
    ("Wi-Fi coverage dead zone in the south wing of floor 4.", "network"),
    ("VLAN misconfiguration — voice traffic not separated from data traffic.", "network"),
    ("SSL certificate expired on the internal web portal — HTTPS warnings.", "network"),
    ("Bandwidth throttling issue — video conferencing quality degraded.", "network"),
    ("New network switch needs configuring for expanded office space.", "network"),
    ("Wireless access point keeps dropping connections after 30 minutes.", "network"),
    ("Network monitoring alerts not sending email notifications.", "network"),
    ("DHCP server not issuing addresses — devices getting APIPA addresses.", "network"),
    ("Proxy server blocking access to approved SaaS application.", "network"),
    ("Site-to-site VPN tunnel keeps dropping every few hours.", "network"),
    ("Port forwarding rules need updating for new external-facing service.", "network"),
    ("Internal network speed test showing 10 Mbps instead of expected 1 Gbps.", "network"),
    ("Remote user cannot access internal file share over VPN.", "network"),
    ("Network cable runs need patching for newly fitted out open plan area.", "network"),
    ("Intrusion detection system generating false positives for internal scanner.", "network"),
    ("Wi-Fi password rotation broke connections on IoT devices.", "network"),
    ("BGP routing table not updated after ISP failover configuration change.", "network"),
    ("Network time protocol drift causing authentication failures on servers.", "network"),

    # ── SOFTWARE (28) ────────────────────────────────────────────────────────
    ("Excel crashing when opening files from the shared drive.", "software"),
    ("Adobe Acrobat licence expired — cannot open PDFs.", "software"),
    ("CRM software throwing database connection error on login.", "software"),
    ("Windows update failed with error code 0x80070005.", "software"),
    ("ERP system running very slowly — reports timing out after 3 minutes.", "software"),
    ("Browser extension causing Teams tab to crash.", "software"),
    ("Antivirus software flagging internal tool as malware — false positive.", "software"),
    ("Python environment not found after IT reimaged the laptop.", "software"),
    ("Software licence not activating — serial key says already in use.", "software"),
    ("macOS upgrade broke compatibility with legacy accounting software.", "software"),
    ("Zoom audio echo issue — microphone feeding back into speaker.", "software"),
    ("Accounting software not exporting reports in the required CSV format.", "software"),
    ("Application deployment failed — MSI installer returns exit code 1603.", "software"),
    ("AutoCAD crashing on opening drawings with external references.", "software"),
    ("Microsoft Teams status stuck on Away despite user being active.", "software"),
    ("Software inventory tool not discovering newly imaged machines.", "software"),
    ("Power BI report failing to refresh — data source credentials error.", "software"),
    ("Slack notifications not appearing on desktop despite app running.", "software"),
    ("Git repository clone failing — SSL certificate verification error.", "software"),
    ("Custom internal app not loading on Chrome, works on Edge.", "software"),
    ("OS disk encryption prompting for recovery key on every reboot.", "software"),
    ("Remote desktop session freezing every 10 minutes on Windows Server.", "software"),
    ("Java runtime version conflict causing ERP module to fail.", "software"),
    ("Task scheduler job not triggering at correct time after DST change.", "software"),
    ("Screen reader accessibility software not compatible with new intranet.", "software"),
    ("VirtualBox VM not starting — error: VT-x disabled in BIOS.", "software"),
    ("Database migration script failed halfway — rollback needed.", "software"),
    ("Office 365 apps showing unlicensed product banner despite valid licence.", "software"),

    # ── STORAGE (29) ────────────────────────────────────────────────────────
    ("Shared drive showing only 2 GB free — team unable to upload files.", "storage"),
    ("Backup job failed last night — alert from the backup monitoring system.", "storage"),
    ("NAS device inaccessible from all workstations since this morning.", "storage"),
    ("User accidentally deleted project folder — need restore from last backup.", "storage"),
    ("OneDrive sync conflict — duplicate files appearing in shared folder.", "storage"),
    ("SAN storage array showing degraded RAID — one disk failed.", "storage"),
    ("File server running out of space — purge or archive policy needed.", "storage"),
    ("Backup tape rotation schedule out of date — tapes not being swapped.", "storage"),
    ("Cloud storage gateway latency high — uploads timing out.", "storage"),
    ("Archive storage tier not accessible through standard explorer path.", "storage"),
    ("Incremental backup taking 8 hours instead of the usual 45 minutes.", "storage"),
    ("Restore from backup completed but files have incorrect permissions.", "storage"),
    ("Storage quota exceeded alert for user — needs increase approval.", "storage"),
    ("Disk deduplication ratio lower than expected — investigate bloat.", "storage"),
    ("Snapshots not being deleted per retention policy — filling up pool.", "storage"),
    ("iSCSI target disconnecting randomly during large file transfers.", "storage"),
    ("FTP server running low on disk space — old logs need clearing.", "storage"),
    ("New project team needs a dedicated shared storage allocation.", "storage"),
    ("Data migration from old file server to new SAN not completed — errors.", "storage"),
    ("Offsite backup replication job failing — network connectivity to DR site.", "storage"),
    ("SharePoint document library hit 5,000 item limit — performance degraded.", "storage"),
    ("Archive email store (.pst) corrupted — cannot open in Outlook.", "storage"),
    ("RAID rebuild taking too long — monitoring shows 4 days remaining.", "storage"),
    ("Backup software agent not installed on newly provisioned server.", "storage"),
    ("User's home drive mapped incorrectly — pointing to another user's folder.", "storage"),
    ("Database transaction logs filling disk — log backup not running.", "storage"),
    ("Cold storage retrieval request for compliance audit — 3-year-old data.", "storage"),
    ("Block storage volume needs to be expanded from 500 GB to 2 TB.", "storage"),
    ("File integrity check failed on archival storage — potential corruption.", "storage"),

        # ACCESS
    ("[INC12345] User unable to login to VPN after password reset, urgent", "access"),
    ("[REQ54321] Please provide access to finance shared folder for new joiner", "access"),
    ("Login issue - account locked again even after unlock from IT", "access"),
    ("Need admin rights on laptop to install Python for project work", "access"),
    ("2FA not working, not receiving OTP on mobile", "access"),

    # EMAIL
    ("Outlook not opening, keeps crashing when clicking on inbox", "email"),
    ("Emails bouncing back to client, getting delivery failure notice", "email"),
    ("Not receiving any emails since morning, pls check", "email"),
    ("Need new email ID for new employee joining tomorrow", "email"),
    ("Phishing mail received, looks suspicious, please investigate", "email"),

    # HARDWARE
    ("Laptop not turning on, no power light", "hardware"),
    ("Keyboard not working properly, some keys stuck", "hardware"),
    ("Monitor showing blank screen after connecting HDMI", "hardware"),
    ("Mouse disconnecting frequently during work", "hardware"),
    ("Laptop overheating and shutting down automatically", "hardware"),

    # HR
    ("New joiner onboarding - need laptop, email and system access", "hr"),
    ("Employee exit process - disable all access by EOD", "hr"),
    ("Payslip not visible in HR portal for last month", "hr"),
    ("Need system access for contractor for 3 months", "hr"),
    ("Attendance not updated correctly in system", "hr"),

    # NETWORK
    ("Internet very slow in office today, unable to work", "network"),
    ("VPN not connecting from home, timeout error", "network"),
    ("WiFi keeps disconnecting every 10 mins", "network"),
    ("Unable to access internal apps over network", "network"),
    ("Network issue in floor 2, multiple users affected", "network"),

    # SOFTWARE
    ("Excel crashing when opening large file", "software"),
    ("Application not launching after update", "software"),
    ("Getting error while logging into CRM tool", "software"),
    ("Software license expired, cannot use tool", "software"),
    ("Teams call audio not working properly", "software"),

    # STORAGE
    ("Shared drive full, cannot upload files", "storage"),
    ("File deleted accidentally, need restore", "storage"),
    ("Backup failed last night, please check", "storage"),
    ("OneDrive not syncing files properly", "storage"),
    ("Need additional storage space for project", "storage")
]

In [3]:
correct = 0
total = len(TEST_CASES)

print("\n--- Test Case Results ---\n")

for i, (text, actual_label) in enumerate(TEST_CASES, 1):
    
    r = predict_ticket(
        text,
        tokenizer,
        model,
        id2label,
    )
    
    predicted_label = r["predicted_category"]
    confidence = r.get("confidence_score", 0.0)
    
    is_correct = predicted_label == actual_label
    if is_correct:
        correct += 1

    print(f"TC{i:03d}")
    print(f"Ticket     : {text}")
    print(f"Actual     : {actual_label}")
    print(f"Predicted  : {predicted_label}")
    print(f"Confidence : {confidence:.2f}")
    print(f"Result     : {'PASS' if is_correct else 'FAIL'}")
    print("-" * 50)

print(f"\nFinal Results: {correct}/{total} correct")
print(f"Accuracy: {correct/total*100:.2f}%")


--- Test Case Results ---

TC001
Ticket     : User cannot log in to the HR portal after password reset.
Actual     : access
Predicted  : access
Confidence : 0.71
Result     : PASS
--------------------------------------------------
TC002
Ticket     : Need to grant read access to the shared finance drive for new employee.
Actual     : access
Predicted  : access
Confidence : 0.80
Result     : PASS
--------------------------------------------------
TC003
Ticket     : Account locked after multiple failed login attempts.
Actual     : access
Predicted  : access
Confidence : 0.81
Result     : PASS
--------------------------------------------------
TC004
Ticket     : Request to revoke access for terminated employee John Smith.
Actual     : access
Predicted  : access
Confidence : 0.83
Result     : PASS
--------------------------------------------------
TC005
Ticket     : Unable to access SharePoint document library — permission denied error.
Actual     : access
Predicted  : storage
Confidence :

In [4]:
TEST_CASES = [
    # ── ACCESS (Focus: Permissions, Credentials, Logins) ──────────────────────
    ("User cannot log in to the HR portal after password reset.", "access"),
    ("Need to grant read access to the shared finance drive for new employee.", "access"),
    ("Account locked after multiple failed login attempts.", "access"),
    ("Request to revoke access for terminated employee John Smith.", "access"),
    ("Unable to access SharePoint document library — permission denied error.", "access"), # Fixed: Permission = Access
    ("New contractor requires temporary access badge and system credentials.", "access"),
    ("Admin rights needed on local machine for software installation.", "access"),
    ("Two-factor authentication not working for VPN login.", "access"),
    ("User forgot their Active Directory password and cannot reset it.", "access"),
    ("Cannot access the payroll system — account shows as inactive.", "access"),
    ("Request elevated privileges for database administrator role.", "access"),
    ("Login session keeps timing out after 5 minutes on the web portal.", "access"),
    ("Single sign-on not redirecting correctly to the CRM dashboard.", "access"),
    ("Service account credentials expired, causing batch jobs to fail.", "access"),
    ("Need to add user to security group for project file access.", "access"),
    ("Guest Wi-Fi access required for visiting vendor in conference room B.", "access"),
    ("Account disabled without notice — user cannot log into any systems.", "access"),
    ("Password policy requires special characters but portal rejects them.", "access"),
    ("User transferred to new department, old access needs updating.", "access"),
    ("Remote desktop connection refused — credentials not accepted.", "access"),
    ("API key for internal tool has expired, request renewal.", "access"),
    ("User needs read-only access to production database for reporting.", "access"),
    ("Cannot unlock workstation — smartcard reader not recognising badge.", "access"), # Note: Could be hardware, but intent is "unlock"
    ("Role-based access not properly applied after user role change in AD.", "access"),
    ("Shared mailbox access request for the support team distribution list.", "access"),
    ("Biometric scanner at server room door not recognising fingerprint.", "access"),
    ("Temporary password sent via SMS not arriving on user's phone.", "access"),
    ("User unable to authenticate to Citrix receiver — error 1030.", "access"),
    ("Access request for new hire starting Monday — all standard applications.", "access"),

    # ── EMAIL (Focus: Outlook, SMTP, Mailboxes, Phishing) ─────────────────────
    ("Outlook keeps crashing when opening attachments larger than 10 MB.", "email"), # Corrected from software
    ("Emails sent to external clients bouncing back with delivery failure.", "email"),
    ("Cannot send emails — SMTP authentication error in Outlook.", "email"),
    ("Inbox not syncing on mobile device after OS upgrade.", "email"),
    ("Spam filter blocking legitimate supplier invoices, need whitelist.", "email"),
    ("Distribution list not delivering to all members.", "email"),
    ("Out-of-office auto-reply not triggering for external senders.", "email"),
    ("Email signature disappeared after recent IT policy update.", "email"),
    ("Shared mailbox missing from Outlook profile after migration.", "email"),
    ("Received phishing email, need IT to investigate and block sender.", "email"),
    ("Calendar invites not syncing between Outlook and Google Calendar.", "email"),
    ("Cannot access archived emails from before January 2023.", "email"),
    ("Outlook search returns no results even for recent emails.", "email"),
    ("Email quota full — old emails cannot be archived automatically.", "email"),
    ("New employee needs corporate email account created.", "email"), 
    ("Rules and filters in Outlook stopped working after update.", "email"),
    ("Email sent but not appearing in sent folder on mobile.", "email"),
    ("Large email attachment limit needs to be increased for project team.", "email"),
    ("Contacts list not populating when typing in the To field.", "email"),
    ("Email account compromised — request immediate password reset and audit.", "email"), # Corrected from access to keep with 'Email'
    ("Delegate access to manager's calendar is not working.", "email"),
    ("Outlook keeps asking for password repeatedly despite correct entry.", "email"),
    ("Need to set up a shared alias address for the customer service team.", "email"),
    ("Emails arriving with garbled characters in subject line (encoding issue).", "email"),
    ("Email forwarding rule set up but forwarded messages not arriving.", "email"),
    ("Unable to open .msg attachments in the web version of Outlook.", "email"),
    ("Migration to Microsoft 365 caused old PST files to become inaccessible.", "email"),
    ("Group email not receiving replies — reply-to address misconfigured.", "email"),
    ("Junk mail folder filling up despite trained spam filter.", "email"),

    # ── HARDWARE (Focus: Physical devices, Peripherals, Power) ────────────────
    ("Laptop screen flickering intermittently, especially on battery power.", "hardware"),
    ("Keyboard keys sticking after liquid spill — needs replacement.", "hardware"),
    ("Printer in room 204 jamming frequently on duplex print jobs.", "hardware"),
    ("Desktop PC not powering on — no lights or fan activity.", "hardware"),
    ("Monitor displaying distorted colours after HDMI cable replacement.", "hardware"),
    ("Docking station not detected when laptop is connected via USB-C.", "hardware"),
    ("Hard drive making clicking noises — suspected imminent failure.", "hardware"),
    ("Wireless mouse losing connection every few minutes.", "hardware"),
    ("Webcam not recognised in any USB port after Windows update.", "hardware"),
    ("Laptop battery draining in under 2 hours despite recent replacement.", "hardware"),
    ("Second monitor not being detected after graphics driver update.", "hardware"),
    ("Barcode scanner in warehouse freezing when scanning high-volume items.", "hardware"),
    ("Headset microphone not working — only one-way audio in Teams calls.", "hardware"),
    ("Desktop RAM needs to be upgraded from 8 GB to 16 GB for design work.", "hardware"),
    ("Printer toner cartridge replaced but still printing faded text.", "hardware"),
    ("Laptop overheating and shutting down during long video calls.", "hardware"),
    ("USB hub on workstation not powering connected devices reliably.", "hardware"),
    ("Trackpad on laptop not responding to multi-finger gestures.", "hardware"),
    ("Network switch in server room showing amber fault light.", "hardware"),
    ("Fingerprint reader on laptop not initialising after Windows Hello setup.", "hardware"),
    ("Scanner flatbed producing vertical lines across all scanned documents.", "hardware"),
    ("Conference room TV remote not pairing with the display unit.", "hardware"),
    ("External SSD not recognised on Mac despite working on Windows.", "hardware"),
    ("Power cable for workstation frayed — fire hazard, needs replacement.", "hardware"),
    ("UPS unit beeping — battery backup may need replacing.", "hardware"),
    ("Laptop hinge cracked — lid wobbles when opening and closing.", "hardware"),
    ("Ethernet port on desktop appears damaged — no physical connection.", "hardware"),
    ("Label printer in dispatch not feeding labels correctly.", "hardware"),
    ("Projector in boardroom not displaying laptop screen via HDMI.", "hardware"),

    # ── HR (Focus: Onboarding, Payroll, Employee Records) ───────────────────
    ("New employee onboarding — need accounts and equipment set up by Monday.", "hr"),
    ("Offboarding request for employee leaving on Friday — revoke all access.", "hr"),
    ("IT system needed for mandatory annual compliance training module.", "hr"),
    ("Employee self-service portal not showing updated payslip for last month.", "hr"),
    ("Maternity leave substitution — temp staff needs system access for 6 months.", "hr"),
    ("Background check system not accessible for new recruitment drive.", "hr"),
    ("HR information system showing incorrect reporting line for new manager.", "hr"),
    ("Request IT equipment for remote worker joining from overseas office.", "hr"),
    ("Performance review system down during appraisal season.", "hr"),
    ("Contractor badge expiry needs extending — still active on project.", "hr"),
    ("Workforce scheduling tool not reflecting updated shift patterns.", "hr"),
    ("New starter laptop needs to be pre-configured with standard software.", "hr"),
    ("Exit interview data not saving in HR portal — form submission error.", "hr"),
    ("Employee attendance system not syncing with payroll software.", "hr"),
    ("HR analytics dashboard permission needed for department manager.", "hr"),
    ("Training platform login credentials not sent to new joiners.", "hr"),
    ("Payroll integration with HR system failing — no data transfer this month.", "hr"),
    ("Time-off request system sending duplicate approval notifications.", "hr"),
    ("Employee records migration to new HR system — data validation errors.", "hr"),
    ("New hire requires IT induction and equipment checklist completion.", "hr"),
    ("Org chart in internal directory not reflecting recent restructure.", "hr"),
    ("Benefits portal showing wrong enrolment status for employee.", "hr"),
    ("HR system audit trail missing entries from last quarter.", "hr"),
    ("Intern accounts need to be created with restricted access profile.", "hr"),
    ("Learning management system not recording completed course credits.", "hr"),
    ("Employee ID badge printing system out of laminate pouches.", "hr"),
    ("Promotion processed in HR system but access rights not updated.", "hr"),
    ("Return-to-work after long-term absence — reactivate dormant account.", "hr"),

    # ── NETWORK (Focus: Wi-Fi, VPN, Connectivity, DNS) ────────────────────────
    ("Office Wi-Fi extremely slow since this morning — affects all users.", "network"),
    ("Cannot connect to VPN from home — connection times out.", "network"),
    ("Intermittent packet loss on wired connection in building 3.", "network"),
    ("DNS resolution failing for internal intranet addresses.", "network"),
    ("Remote site connectivity down — branch office cannot reach HQ systems.", "network"),
    ("Firewall blocking legitimate traffic to cloud storage provider.", "network"),
    ("IP address conflict causing network drop for multiple workstations.", "network"),
    ("Network printer offline despite being powered on and connected.", "network"),
    ("VPN client showing connected but no traffic passing through tunnel.", "network"),
    ("Load balancer misconfiguration causing intermittent 502 errors.", "network"),
    ("Wi-Fi coverage dead zone in the south wing of floor 4.", "network"),
    ("VLAN misconfiguration — voice traffic not separated from data traffic.", "network"),
    ("SSL certificate expired on the internal web portal — HTTPS warnings.", "network"),
    ("Bandwidth throttling issue — video conferencing quality degraded.", "network"),
    ("New network switch needs configuring for expanded office space.", "network"),
    ("Wireless access point keeps dropping connections after 30 minutes.", "network"),
    ("Network monitoring alerts not sending email notifications.", "network"),
    ("DHCP server not issuing addresses — devices getting APIPA addresses.", "network"),
    ("Proxy server blocking access to approved SaaS application.", "network"),
    ("Site-to-site VPN tunnel keeps dropping every few hours.", "network"),
    ("Port forwarding rules need updating for new external-facing service.", "network"),
    ("Internal network speed test showing 10 Mbps instead of expected 1 Gbps.", "network"),
    ("Remote user cannot access internal file share over VPN.", "network"),
    ("Network cable runs need patching for newly fitted out open plan area.", "network"),
    ("Intrusion detection system generating false positives for internal scanner.", "network"),
    ("Wi-Fi password rotation broke connections on IoT devices.", "network"),
    ("BGP routing table not updated after ISP failover configuration change.", "network"),
    ("Network time protocol drift causing authentication failures on servers.", "network"),

    # ── SOFTWARE (Focus: Apps, OS, Licences, Crashing) ────────────────────────
    ("Excel crashing when opening files from the shared drive.", "software"),
    ("Adobe Acrobat licence expired — cannot open PDFs.", "software"),
    ("CRM software throwing database connection error on login.", "software"),
    ("Windows update failed with error code 0x80070005.", "software"),
    ("ERP system running very slowly — reports timing out after 3 minutes.", "software"),
    ("Browser extension causing Teams tab to crash.", "software"),
    ("Antivirus software flagging internal tool as malware — false positive.", "software"),
    ("Python environment not found after IT reimaged the laptop.", "software"),
    ("Software licence not activating — serial key says already in use.", "software"),
    ("macOS upgrade broke compatibility with legacy accounting software.", "software"),
    ("Zoom audio echo issue — microphone feeding back into speaker.", "software"),
    ("Accounting software not exporting reports in the required CSV format.", "software"),
    ("Application deployment failed — MSI installer returns exit code 1603.", "software"),
    ("AutoCAD crashing on opening drawings with external references.", "software"),
    ("Microsoft Teams status stuck on Away despite user being active.", "software"),
    ("Software inventory tool not discovering newly imaged machines.", "software"),
    ("Power BI report failing to refresh — data source credentials error.", "software"),
    ("Slack notifications not appearing on desktop despite app running.", "software"),
    ("Git repository clone failing — SSL certificate verification error.", "software"),
    ("Custom internal app not loading on Chrome, works on Edge.", "software"),
    ("OS disk encryption prompting for recovery key on every reboot.", "software"),
    ("Remote desktop session freezing every 10 minutes on Windows Server.", "software"),
    ("Java runtime version conflict causing ERP module to fail.", "software"),
    ("Task scheduler job not triggering at correct time after DST change.", "software"),
    ("Screen reader accessibility software not compatible with new intranet.", "software"),
    ("VirtualBox VM not starting — error: VT-x disabled in BIOS.", "software"),
    ("Database migration script failed halfway — rollback needed.", "software"),
    ("Office 365 apps showing unlicensed product banner despite valid licence.", "software"),

    # ── STORAGE (Focus: Disk Space, Backups, NAS, Cloud Storage) ──────────────
    ("Shared drive showing only 2 GB free — team unable to upload files.", "storage"),
    ("Backup job failed last night — alert from the backup monitoring system.", "storage"),
    ("NAS device inaccessible from all workstations since this morning.", "storage"),
    ("User accidentally deleted project folder — need restore from last backup.", "storage"),
    ("OneDrive sync conflict — duplicate files appearing in shared folder.", "storage"),
    ("SAN storage array showing degraded RAID — one disk failed.", "storage"),
    ("File server running out of space — purge or archive policy needed.", "storage"),
    ("Backup tape rotation schedule out of date — tapes not being swapped.", "storage"),
    ("Cloud storage gateway latency high — uploads timing out.", "storage"),
    ("Archive storage tier not accessible through standard explorer path.", "storage"),
    ("Incremental backup taking 8 hours instead of the usual 45 minutes.", "storage"),
    ("Restore from backup completed but files have incorrect permissions.", "storage"),
    ("Storage quota exceeded alert for user — needs increase approval.", "storage"),
    ("Disk deduplication ratio lower than expected — investigate bloat.", "storage"),
    ("Snapshots not being deleted per retention policy — filling up pool.", "storage"),
    ("iSCSI target disconnecting randomly during large file transfers.", "storage"),
    ("FTP server running low on disk space — old logs need clearing.", "storage"),
    ("New project team needs a dedicated shared storage allocation.", "storage"),
    ("Data migration from old file server to new SAN not completed — errors.", "storage"),
    ("Offsite backup replication job failing — network connectivity to DR site.", "storage"),
    ("SharePoint document library hit 5,000 item limit — performance degraded.", "storage"),
    ("Archive email store (.pst) corrupted — cannot open in Outlook.", "storage"),
    ("RAID rebuild taking too long — monitoring shows 4 days remaining.", "storage"),
    ("Backup software agent not installed on newly provisioned server.", "storage"),
    ("User's home drive mapped incorrectly — pointing to another user's folder.", "storage"),
    ("Database transaction logs filling disk — log backup not running.", "storage"),
    ("Cold storage retrieval request for compliance audit — 3-year-old data.", "storage"),
    ("Block storage volume needs to be expanded from 500 GB to 2 TB.", "storage"),
    ("File integrity check failed on archival storage — potential corruption.", "storage")
]

In [5]:
correct = 0
total = len(TEST_CASES)

print("\n--- Test Case Results ---\n")

for i, (text, actual_label) in enumerate(TEST_CASES, 1):
    
    r = predict_ticket(
        text,
        tokenizer,
        model,
        id2label,
    )
    
    predicted_label = r["predicted_category"]
    confidence = r.get("confidence_score", 0.0)
    
    is_correct = predicted_label == actual_label
    if is_correct:
        correct += 1

    print(f"TC{i:03d}")
    print(f"Ticket     : {text}")
    print(f"Actual     : {actual_label}")
    print(f"Predicted  : {predicted_label}")
    print(f"Confidence : {confidence:.2f}")
    print(f"Result     : {'PASS' if is_correct else 'FAIL'}")
    print("-" * 50)

print(f"\nFinal Results: {correct}/{total} correct")
print(f"Accuracy: {correct/total*100:.2f}%")


--- Test Case Results ---

TC001
Ticket     : User cannot log in to the HR portal after password reset.
Actual     : access
Predicted  : access
Confidence : 0.71
Result     : PASS
--------------------------------------------------
TC002
Ticket     : Need to grant read access to the shared finance drive for new employee.
Actual     : access
Predicted  : access
Confidence : 0.80
Result     : PASS
--------------------------------------------------
TC003
Ticket     : Account locked after multiple failed login attempts.
Actual     : access
Predicted  : access
Confidence : 0.81
Result     : PASS
--------------------------------------------------
TC004
Ticket     : Request to revoke access for terminated employee John Smith.
Actual     : access
Predicted  : access
Confidence : 0.83
Result     : PASS
--------------------------------------------------
TC005
Ticket     : Unable to access SharePoint document library — permission denied error.
Actual     : access
Predicted  : storage
Confidence :